# VWAPSb backtest: compare model parameters

For each selected observed physical contract, hide that observation and all equivalent aliases on that date, then predict it using the remaining own data and the allowed EEX information. Repeat for every candidate configuration. Earlier own history remains available; future own prices are excluded.

**No automatic winner is chosen.** Compare errors, coverage and sample size by unit, product and contract family. This tests isolated missing observations, not several simultaneous or persistent gaps. Start with the invented demo; success on synthetic data does not demonstrate accuracy on your real curves.


In [ ]:
PROJECT_ROOT = None  # Example: r"C:\Projects\VWAPSb".
CONFIG_PATH = None  # An explicit config overrides USE_DEMO.
USE_DEMO = True  # Create a separate invented dataset when CONFIG_PATH is None.
START_DATE = None  # Inclusive ISO date, e.g. "2026-09-21"; None uses first own observation.
END_DATE = None  # Inclusive ISO date, e.g. "2026-10-09"; None uses last own observation.
EEX_OFFSET_DAYS = None  # Keep config; 0 permits same-day publication, -1 excludes it.
OUTPUT_DIR = None  # Default: selected config output_dir / "backtest".
MASK_FRACTION = 1.0  # Fraction of cases examined ONE AT A TIME; not simultaneous hiding.
SEED = 42


In [ ]:
# Candidate lists form a Cartesian grid. Omitted fields retain each product's
# effective fixed configuration. At most 256 candidates are allowed.
PARAMETER_GRID = {
    "local_max_steps": [1, 2],
    "history_enabled": [False, True],
}
# Other examples:
# PARAMETER_GRID = {"eex_average": ["simple", "ewma"], "eex_window": [3, 5, 9]}
# PARAMETER_GRID = {"local_enabled": [False], "history_enabled": [False],
#                   "eex_average": ["simple", "ewma"], "eex_halflife": [1.0, 2.0]}
# PARAMETER_GRID = {}  # Evaluate only the current effective configuration.


In [ ]:
from pathlib import Path
import sys
from dataclasses import replace
from datetime import date

import pandas as pd
from IPython.display import display

if PROJECT_ROOT is None:
    current = Path.cwd().resolve()
    ROOT = next((path for path in [current, *current.parents] if (path / "src/vwapsb").is_dir()), None)
    if ROOT is None:
        raise ValueError("Set PROJECT_ROOT to the VWAPSb repository directory.")
else:
    ROOT = Path(PROJECT_ROOT).expanduser().resolve()
if not (ROOT / "src/vwapsb").is_dir():
    raise ValueError("PROJECT_ROOT must contain src/vwapsb.")
sys.path.insert(0, str(ROOT / "src")) if str(ROOT / "src") not in sys.path else None

def project_path(value):
    """Resolve notebook paths relative to the repository root."""
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (ROOT / path).resolve()

from vwapsb.config import load_config
from vwapsb.viewer import create_viewer, load_output

from vwapsb.backtest import configured_candidates, run_backtest
from vwapsb.demo import create_demo
from vwapsb.io_vwap import load_vwaps
from vwapsb.io_eex import load_eex_books
from vwapsb.mapping import load_mapping

if CONFIG_PATH is not None:
    config_path = project_path(CONFIG_PATH)
elif USE_DEMO:
    config_path = create_demo(ROOT)
else:
    config_path = ROOT / "config.local.toml" if (ROOT / "config.local.toml").is_file() else ROOT / "config.toml"
config = load_config(config_path)
if EEX_OFFSET_DAYS is not None:
    config = replace(config, eex_offset_days=EEX_OFFSET_DAYS)
mappings = load_mapping(config)
observations = load_vwaps(config.vwap_input, config)
books = load_eex_books(config, mappings)
start = observations.date.min() if START_DATE is None else date.fromisoformat(START_DATE)
end = observations.date.max() if END_DATE is None else date.fromisoformat(END_DATE)
candidates = configured_candidates(config, mappings, PARAMETER_GRID)
print(f"Config: {config_path}")
print(f"Dates: {start} to {end}; EEX calendar-day offset: {config.eex_offset_days}")
print(f"Candidates: {len(candidates)}; examined case fraction: {MASK_FRACTION}")
display(pd.DataFrame(candidates))


The fixed examination sample contains finite own observations in active mapped curves, even when a candidate rejects them as usable anchors. Duplicate observations for one physical period are consolidated using positive volumes; without positive volumes, their simple mean is the hidden actual. Each case is restored before examining another case.

`MASK_FRACTION=0.5` samples approximately half of these cases reproducibly. It still hides one physical period at a time. All candidates use the same selected cases. `START_DATE` and `END_DATE` restrict evaluated cases; earlier supplied observations can still provide historical evidence.

In individual production mode, non-varied parameters retain each product's effective configuration. Grid values override those fixed values for the experiment only. No configuration or production output is modified.


In [ ]:
reports = run_backtest(
    observations, books, mappings, config, start, end,
    grid=PARAMETER_GRID, mask_fraction=MASK_FRACTION, seed=SEED,
)
report_dir = project_path(OUTPUT_DIR) if OUTPUT_DIR is not None else config.resolve(config.output_dir) / "backtest"
report_dir.mkdir(parents=True, exist_ok=True)
for name, frame in reports.items():
    frame.to_csv(report_dir / f"{name}.csv", index=False, encoding="utf-8-sig")
print(f"Saved backtest CSVs to {report_dir}")


`coverage = n_predicted / n_cases` includes all examined cases in its denominator. MAE is the mean absolute price error, RMSE penalizes larger errors more strongly, and bias is the mean signed error (`prediction - actual`). These errors use only finite predictions. A candidate that predicts fewer difficult cases can look better on MAE, so coverage and counts must be considered alongside errors.

Units remain separate: EUR/MWh and GBP/MWh errors are never averaged together. The detailed table also separates products and contract families. There is no universal best configuration or hidden score that automatically ranks these trade-offs. If you change the grid after inspecting these dates, evaluate the final choice on a later untouched interval.


In [ ]:
display(reports["summary"])
display(reports["by_curve"])


Select a candidate, product and date below to inspect its hidden-observation predictions against EEX and the actual hidden prices. These charts show the backtest cases, not a complete candidate refill. For complete curves after choosing settings manually, run refill and open `inspect_curves.ipynb`.


In [ ]:
viewer = create_viewer(reports["scores"])
display(viewer)
